# Impedance ML Engineering — Session 13
## Learn representations with masked autoencoders
**Core: 60 minutes · Neural networks · Optional extension: 20–40 minutes**

## Why this matters
Can unsupervised compression capture useful matrix structure without learning QC placeholders?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Train an autoencoder with observed-cell reconstruction loss.
- Compare reconstruction with a PCA baseline.
- Inspect reconstruction maps and missingness dependence.
- Separate reconstruction ability from predictive value.

## R-to-Python / prerequisite bridge
PCA inverse transform → nonlinear encoder/decoder; reconstruction loss needs an explicit mask and observed-cell denominator.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
An autoencoder compresses inputs through a bottleneck and reconstructs them. It can learn nonlinear structure, but it can also learn measurement artifacts or waste capacity on high-variance coordinates. For masked data, the loss must compare only truly observed cells with a valid denominator.

A filled invalid target must be finite before multiplication by the mask: NaN×0 is still NaN. A global observed-cell objective weights matrices by their observed-cell counts; a mean of per-matrix losses weights visits equally. State which objective you choose. A denoising input mask and an original target-validity mask serve different purposes.

Use `nn.Linear` encoder/decoder blocks, `.sum()` for numerator/denominator, and eval/no_grad when exporting a frozen embedding. Compare PCA and autoencoder residuals in the same space.

## Task 1 — Build a reconstruction dataset (6 min)
Use current matrices only with fixed patient train/validation partitions. Apply training-only global observed-cell standardization and retain masks. Flatten values/masks; target is the same current standardized matrix, not next-visit outcome. Predict input shape (N, 288) and output shape (N, 144).

## Task 2 — Define a compact autoencoder (8 min)
Use encoder Linear(288, 32)→ReLU→Linear(32, 8), decoder Linear(8, 32)→ReLU→Linear(32, 144). Count parameters and inspect a forward output. Provide the mask in the input so a filled placeholder is distinguishable from a valid centered reading.

## Task 3 — Implement masked MSE (8 min)
Compute `((reconstruction-target)**2 * mask).sum()/mask.sum()` with targets filled finite at invalid cells. Do not multiply NaN by 0: it stays NaN. Assert denominator>0. Change targets only at masked cells and verify loss is unchanged; predict the scalar loss shape.

## Task 4 — Train against reconstruction validation (10 min)
Use Adam(lr=0.005), batch_size 32, max 60 epochs, seed 42, patience 8 based on validation observed-cell reconstruction MSE. Aggregate epoch losses by observed-cell counts, not equally across batches. Restore the best checkpoint. Never evaluate on the reserved test.

## Task 5 — Compare with PCA (8 min)
Fit imputation/scaling/PCA with 8 components on training only. Inverse-transform validation reconstructions back into the SAME globally standardized value space as the autoencoder. Evaluate both only on original valid cells. Explain why comparing errors in different scales or on imputed versus observed cells is unfair.

## Task 6 — Inspect three reconstructions (6 min)
Select validation matrices with low/median/high observed-cell error and plot original, reconstruction, residual, and mask. Inspect diagonal/off-diagonal errors separately. Do not label a high error as disease; it may be a measurement artifact or unusual coverage.

## Task 7 — Probe embeddings for prediction (6 min)
Freeze the trained encoder; extract train/validation 8-dimensional embeddings with eval/no_grad. Fit Ridge(alpha=1) on training embeddings to predict next-visit diagonal mean. Compare persistence and a summary ridge. Predict the embedding table’s grain and explain why unsupervised preprocessing still must avoid evaluation patients.

## Task 8 — Write the representation conclusion (0–2 min)
Report reconstruction versus downstream predictive comparisons, scaling/mask policy, and one limitation. Explain why good reconstruction does not guarantee target-relevant features.

## Retrieval practice
Write the difference between a validity mask used in a loss and a separate input mask channel; neither replaces the other.

## Professional reasoning
1. What could go wrong if the reconstruction loss is dominated by the large diagonal?
2. Why should target outcomes stay unused during unsupervised representation learning?
3. How would independently annotated anomalies change the evaluation?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Randomly hide 10% of training-valid cells, preserving the original target-valid mask, and train a denoising version. Carefully distinguish input visibility from valid reconstruction targets.

## API references
[PyTorch training basics](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html) · [Conv2d reference](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html)